In [26]:
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ── Top 10 S&P 500 by market cap (current)
TOP10 = ['AAPL','MSFT','NVDA','AMZN','GOOGL','META','TSLA','AVGO','LLY','JPM']

# ── Pull quarterly EPS (actual) — need 16 quarters to compute rolling σ properly
#    get_earnings_dates gives EPS Estimate + EPS Actual per earnings release
N_QUARTERS = 20   # fetch plenty of history

print("Fetching quarterly EPS...")
eps_dict = {}
for ticker in TOP10:
    try:
        t  = yf.Ticker(ticker)
        ed = t.get_earnings_dates(limit=N_QUARTERS)
        if ed is None or ed.empty:
            print(f"  {ticker}: no data")
            continue
        ed = ed.dropna(subset=['Reported EPS']).sort_index()
        ed.index = pd.to_datetime(ed.index).tz_localize(None)
        eps_dict[ticker] = ed['Reported EPS'].sort_index()
        print(f"  {ticker}: {len(eps_dict[ticker])} quarters  "
              f"({eps_dict[ticker].index[0].date()} → {eps_dict[ticker].index[-1].date()})")
    except Exception as e:
        print(f"  {ticker}: ERROR — {e}")

print(f"\nGot EPS for {len(eps_dict)}/{len(TOP10)} tickers")

Fetching quarterly EPS...
  AAPL: 24 quarters  (2020-04-30 → 2026-01-29)
  MSFT: 24 quarters  (2020-04-29 → 2026-01-28)
  NVDA: 24 quarters  (2020-05-21 → 2026-02-25)
  AMZN: 24 quarters  (2020-04-30 → 2026-02-05)
  GOOGL: 24 quarters  (2020-04-28 → 2026-02-04)
  META: 24 quarters  (2020-04-29 → 2026-01-28)
  TSLA: 24 quarters  (2020-04-29 → 2026-01-28)
  AVGO: 24 quarters  (2020-06-04 → 2026-03-04)
  LLY: 24 quarters  (2020-04-23 → 2026-02-04)
  JPM: 24 quarters  (2020-04-14 → 2026-01-13)

Got EPS for 10/10 tickers


In [27]:
from IPython.display import display

# Need 24 quarters:
#   8 deltas for c/σ window  → requires 12 quarters
#   + 4 quarters warmup (min 4 deltas before first valid SUE)
#   = 16 minimum, but 24 gives clean full history with no truncation
N = 24
aligned = {}
for ticker, s in eps_dict.items():
    s_clean = s.sort_index().dropna()
    aligned[ticker] = s_clean.values[-N:] if len(s_clean) >= N else s_clean.values

eps_panel = pd.DataFrame(aligned)
eps_panel.index = range(-len(eps_panel)+1, 1)   # -23, -22, ..., 0
eps_panel.index.name = 'Quarter (0=latest)'

# Show t through t-4
display_df = eps_panel.loc[-4:0].copy()
display_df.index = ['t-4','t-3','t-2','t-1','t']
display_df.index.name = 'Quarter'

print("Reported EPS — t through t-4\n")
display(display_df.style
        .format("{:.4f}", na_rep="—")
        .background_gradient(cmap='RdYlGn', axis=None)
        .set_caption("<b>Reported EPS</b>"))

Reported EPS — t through t-4



,AAPL,MSFT,NVDA,AMZN,GOOGL,META,TSLA,AVGO,LLY,JPM
Quarter,,,,,,,,,,
t-4,2.4000,3.2300,0.8900,1.8600,2.1500,8.0200,0.7300,1.6000,5.3200,4.8100
t-3,1.6500,3.4600,0.8100,1.5900,2.8100,6.4300,0.1200,1.5800,3.0600,5.0700
t-2,1.5700,3.6500,1.0500,1.6800,2.3100,7.1400,0.4000,1.6900,6.3100,4.9600
t-1,1.8500,3.7200,1.3000,1.9500,2.8700,7.2500,0.3900,1.9500,7.0200,5.0700
t,2.8400,5.1600,1.6200,1.9500,2.8200,8.8800,0.2400,1.5000,7.5400,4.6300


In [28]:
# ── SUE Computation ───────────────────────────────────────────────────────────
# ΔE(t) = E(t) - E(t-4)   (seasonal random walk)
# c(j,t), σ(j,t) = mean/std of ΔE over preceding 8 quarters (min 4)
# SUE(j,t) = (ΔE(t) - c(j,t)) / σ(j,t)

sue_dict   = {}
delta_dict = {}

for ticker in eps_panel.columns:
    s = eps_panel[ticker].dropna()
    if len(s) < 5:
        continue

    deltas = {}
    for i in s.index:
        if i - 4 in s.index and pd.notna(s[i]) and pd.notna(s[i-4]):
            deltas[i] = s[i] - s[i-4]

    delta_s = pd.Series(deltas, name=ticker)
    delta_dict[ticker] = delta_s

    sue_vals = {}
    for i in sorted(delta_s.index):
        window = delta_s[delta_s.index < i].tail(8)
        if len(window) < 4:
            sue_vals[i] = np.nan
        else:
            c   = window.mean()
            sig = window.std(ddof=1)
            sue_vals[i] = (delta_s[i] - c) / sig if sig != 0 else np.nan

    sue_dict[ticker] = pd.Series(sue_vals, name=ticker)

delta_panel = pd.DataFrame(delta_dict)
sue_panel   = pd.DataFrame(sue_dict)

print("ΔE panel (last 4 quarters):")
display(delta_panel.tail(4).style.format("{:.4f}", na_rep="—")
        .background_gradient(cmap='RdYlGn', axis=None))

print("\nRaw SUE (last 4 quarters):")
display(sue_panel.tail(4).style.format("{:.3f}", na_rep="—")
        .background_gradient(cmap='RdYlGn', axis=None))

ΔE panel (last 4 quarters):


,AAPL,MSFT,NVDA,AMZN,GOOGL,META,TSLA,AVGO,LLY,JPM
-3,0.1200,0.5200,0.2000,0.6100,0.9200,1.7200,-0.3300,0.4800,0.4800,0.6300
-2,0.1700,0.7000,0.3700,0.4200,0.4200,1.9800,-0.1200,0.4500,2.3900,-1.1600
-1,0.2100,0.4200,0.4900,0.5200,0.7500,1.2200,-0.3300,0.5300,5.8400,0.7000
0,0.4400,1.9300,0.7300,0.0900,0.6700,0.8600,-0.4900,-0.1000,2.2200,-0.1800



Raw SUE (last 4 quarters):


,AAPL,MSFT,NVDA,AMZN,GOOGL,META,TSLA,AVGO,LLY,JPM
-3,-0.143,0.501,-0.786,-0.736,1.971,-0.148,-0.500,1.949,-0.102,-0.364
-2,0.221,1.405,0.096,-1.830,-0.694,-0.237,0.398,1.229,1.168,-2.288
-1,0.557,-0.393,1.280,-0.783,1.007,-1.818,-0.364,1.365,3.315,0.242
0,3.207,8.176,3.563,-2.959,0.317,-1.812,-1.190,-2.202,0.138,-0.602


In [29]:
# ── Cross-Sectional Normalization at time t ───────────────────────────────────
latest_sue = sue_panel.iloc[-1].dropna()

sue_z    = (latest_sue - latest_sue.mean()) / latest_sue.std(ddof=1)
sue_rank = latest_sue.rank(method='average').astype(int)

signal = pd.DataFrame({
    'EPS(t)':       eps_panel.loc[0,  latest_sue.index],
    'EPS(t-4)':     eps_panel.loc[-4, latest_sue.index],
    'ΔE(t)':        delta_panel.iloc[-1][latest_sue.index],
    'Raw SUE':      latest_sue,
    'SUE z-score':  sue_z,
    'Rank (1=low)': sue_rank,
}, index=latest_sue.index)

signal = signal.sort_values('SUE z-score', ascending=False)
signal.index.name = 'Ticker'

print(f"Signal for t+1 trade  |  {len(signal)} stocks\n")
display(signal.style
        .format({'EPS(t)': '{:.4f}', 'EPS(t-4)': '{:.4f}',
                 'ΔE(t)': '{:.4f}', 'Raw SUE': '{:.3f}',
                 'SUE z-score': '{:.3f}', 'Rank (1=low)': '{:.0f}'}, na_rep='—')
        .background_gradient(subset=['SUE z-score'],  cmap='RdYlGn')
        .background_gradient(subset=['Rank (1=low)'], cmap='RdYlGn')
        .set_caption("<b>SUE Signal — top = long, bottom = short candidates</b>"))

Signal for t+1 trade  |  10 stocks



,EPS(t),EPS(t-4),ΔE(t),Raw SUE,SUE z-score,Rank (1=low)
Ticker,,,,,,
MSFT,5.1600,3.2300,1.9300,8.176,2.208,10
NVDA,1.6200,0.8900,0.7300,3.563,0.852,9
AAPL,2.8400,2.4000,0.4400,3.207,0.747,8
GOOGL,2.8200,2.1500,0.6700,0.317,-0.102,7
LLY,7.5400,5.3200,2.2200,0.138,-0.155,6
JPM,4.6300,4.8100,-0.1800,-0.602,-0.372,5
TSLA,0.2400,0.7300,-0.4900,-1.190,-0.545,4
META,8.8800,8.0200,0.8600,-1.812,-0.727,3
AVGO,1.5000,1.6000,-0.1000,-2.202,-0.842,2


In [30]:
import statsmodels.api as sm

# ── Step 1: Reconstruct actual earnings dates panel (same N=24 alignment as eps_panel)
N = 24
dates_aligned = {}
for ticker, s in eps_dict.items():
    s_clean = s.sort_index().dropna()
    dates_aligned[ticker] = s_clean.index[-N:].tolist() if len(s_clean) >= N else s_clean.index.tolist()

# Build index explicitly — do NOT rely on eps_panel.index in case it's stale
dates_panel = pd.DataFrame(dates_aligned)
dates_panel.index = range(-len(dates_panel)+1, 1)
dates_panel.index.name = 'Quarter (0=latest)'

# ── Step 2: Download prices
earliest = min(d for row in dates_panel.values for d in row if pd.notna(d))
price_start = (pd.Timestamp(earliest) - pd.DateOffset(months=8)).strftime('%Y-%m-%d')
price_end   = pd.Timestamp.today().strftime('%Y-%m-%d')

prices = yf.download(TOP10, start=price_start, end=price_end,
                     auto_adjust=True, progress=False)['Close']
prices = prices.ffill()
log_rets = np.log(prices / prices.shift(1))

print(f"Prices downloaded: {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"dates_panel shape: {dates_panel.shape}  |  eps_panel shape: {eps_panel.shape}")

# ── Step 3: 6M returns
ret6m_panel = pd.DataFrame(index=dates_panel.index, columns=TOP10, dtype=float)

for ticker in TOP10:
    if ticker not in prices.columns:
        continue
    for q in dates_panel.index:
        ann_date = dates_panel.loc[q, ticker]
        if pd.isna(ann_date):
            continue
        ann_date = pd.Timestamp(ann_date)
        avail = prices.index[prices.index <= ann_date]
        if len(avail) < 127:
            continue
        t_idx   = avail[-1]
        t6m_idx = avail[-127]
        ret6m_panel.loc[q, ticker] = float(log_rets.loc[t6m_idx:t_idx, ticker].sum())

print("\n6M return panel (last 4 quarters):")
display(ret6m_panel.tail(4).astype(float).style
        .format("{:.4f}", na_rep="—")
        .background_gradient(cmap='RdYlGn', axis=None)
        .set_caption("<b>Past 6-Month Log Returns</b>"))

Prices downloaded: 2019-08-14 → 2026-03-04
dates_panel shape: (24, 10)  |  eps_panel shape: (24, 10)

6M return panel (last 4 quarters):


,AAPL,MSFT,NVDA,AMZN,GOOGL,META,TSLA,AVGO,LLY,JPM
Quarter (0=latest),,,,,,,,,,
-3,-0.0792,-0.0679,-0.0788,0.0125,-0.0234,-0.0319,0.0725,0.4780,-0.1137,0.1253
-2,-0.1355,0.1702,0.3607,-0.0171,-0.0275,0.0538,-0.2488,0.4956,-0.2501,0.1753
-1,0.2471,0.3215,0.3280,0.1893,0.5411,0.3058,0.6123,0.4783,-0.0582,0.2557
0,0.2030,-0.0586,0.0943,0.0410,0.5366,-0.0691,0.2815,0.0692,0.3681,0.0834


In [31]:
# ── Step 4: Cross-sectional regression SUE ~ Ret6M at each quarter t
# Adj.SUE(j,t) = residual of:  SUE(j,t) = α(t) + β(t)*Ret6M(j,t) + ε(j,t)

adj_sue_dict = {}

for q in sue_panel.index:
    sue_row  = sue_panel.loc[q].dropna()
    ret_row  = ret6m_panel.loc[q].dropna() if q in ret6m_panel.index else pd.Series(dtype=float)

    # Need both SUE and Ret6M for the same tickers
    common = sue_row.index.intersection(ret_row.index)
    if len(common) < 4:   # need at least 4 stocks to run regression meaningfully
        adj_sue_dict[q] = pd.Series(np.nan, index=sue_row.index)
        continue

    y = sue_row[common].values.reshape(-1, 1)
    X = ret_row[common].values.reshape(-1, 1)

    reg = LinearRegression().fit(X, y)
    resid = (y - reg.predict(X)).flatten()

    adj_sue_dict[q] = pd.Series(resid, index=common)

adj_sue_panel = pd.DataFrame(adj_sue_dict).T   # quarters × tickers

print(f"Adj. SUE panel  shape: {adj_sue_panel.shape}")
print(f"Cross-sec R² by quarter (how much momentum explains SUE):")
r2_series = {}
for q in sue_panel.index:
    sue_row = sue_panel.loc[q].dropna()
    ret_row = ret6m_panel.loc[q].dropna() if q in ret6m_panel.index else pd.Series(dtype=float)
    common  = sue_row.index.intersection(ret_row.index)
    if len(common) < 4:
        continue
    y = sue_row[common].values
    X = ret_row[common].values.reshape(-1,1)
    r2_series[q] = LinearRegression().fit(X, y).score(X, y)
print(pd.Series(r2_series).round(3).to_string())

print("\nAdj. SUE panel (last 4 quarters):")
display(adj_sue_panel.tail(4).astype(float).style
        .format("{:.3f}", na_rep="—")
        .background_gradient(cmap='RdYlGn', axis=None)
        .set_caption("<b>Momentum-Adjusted SUE (regression residuals)</b>"))

Adj. SUE panel  shape: (20, 10)
Cross-sec R² by quarter (how much momentum explains SUE):
-15    0.090
-14    0.012
-13    0.028
-12    0.033
-11    0.028
-10    0.025
-9     0.108
-8     0.085
-7     0.310
-6     0.142
-5     0.020
-4     0.045
-3     0.255
-2     0.002
-1     0.121
 0     0.020

Adj. SUE panel (last 4 quarters):


,AAPL,AMZN,AVGO,GOOGL,JPM,LLY,META,MSFT,NVDA,TSLA
-3,0.012,-0.851,0.465,1.962,-0.811,0.154,-0.132,0.623,-0.632,-0.791
-2,0.320,-1.759,1.179,-0.621,-2.263,1.294,-0.183,1.432,0.078,0.523
-1,-0.077,-1.566,1.327,1.131,-0.370,1.893,-2.301,-0.835,0.854,-0.057
0,2.663,-3.906,-3.079,0.605,-1.444,0.005,-3.034,6.980,2.748,-1.538


In [32]:
# ── Step 5: Normalize Adj. SUE cross-sectionally at t (same as raw SUE)
latest_adj = adj_sue_panel.iloc[-1].dropna()

adj_z    = (latest_adj - latest_adj.mean()) / latest_adj.std(ddof=1)
adj_rank = latest_adj.rank(method='average').astype(int)

# Pull raw SUE signal for side-by-side comparison
latest_sue = sue_panel.iloc[-1].dropna()
sue_z      = (latest_sue - latest_sue.mean()) / latest_sue.std(ddof=1)

all_tickers = adj_z.index.union(sue_z.index)

adj_signal = pd.DataFrame({
    'Ret 6M':          ret6m_panel.iloc[-1][all_tickers],
    'Raw SUE':         latest_sue.reindex(all_tickers),
    'SUE z-score':     sue_z.reindex(all_tickers),
    'Adj.SUE resid':   latest_adj.reindex(all_tickers),
    'Adj.SUE z-score': adj_z.reindex(all_tickers),
    'Adj.SUE rank':    adj_rank.reindex(all_tickers),
}, index=all_tickers)

adj_signal = adj_signal.sort_values('Adj.SUE z-score', ascending=False)
adj_signal.index.name = 'Ticker'

print("SUE vs Momentum-Adjusted SUE signal at t\n")
display(adj_signal.style
        .format({'Ret 6M': '{:.3f}', 'Raw SUE': '{:.3f}', 'SUE z-score': '{:.3f}',
                 'Adj.SUE resid': '{:.3f}', 'Adj.SUE z-score': '{:.3f}',
                 'Adj.SUE rank': '{:.0f}'}, na_rep='—')
        .background_gradient(subset=['SUE z-score'],     cmap='RdYlGn')
        .background_gradient(subset=['Adj.SUE z-score'], cmap='RdYlGn')
        .background_gradient(subset=['Ret 6M'],          cmap='RdYlBu')
        .set_caption("<b>Adj. SUE Signal — momentum-orthogonalized, sorted by Adj.SUE z-score</b>"))

SUE vs Momentum-Adjusted SUE signal at t



,Ret 6M,Raw SUE,SUE z-score,Adj.SUE resid,Adj.SUE z-score,Adj.SUE rank
Ticker,,,,,,
MSFT,-0.059,8.176,2.208,6.980,2.072,10
NVDA,0.094,3.563,0.852,2.748,0.816,9
AAPL,0.203,3.207,0.747,2.663,0.791,8
GOOGL,0.537,0.317,-0.102,0.605,0.180,7
LLY,0.368,0.138,-0.155,0.005,0.002,6
JPM,0.083,-0.602,-0.372,-1.444,-0.429,5
TSLA,0.282,-1.190,-0.545,-1.538,-0.457,4
META,-0.069,-1.812,-0.727,-3.034,-0.901,3
AVGO,0.069,-2.202,-0.842,-3.079,-0.914,2


In [33]:
import statsmodels.api as sm

# ── Pre-earnings returns: 7, 14, 21 trading days before announcement
WINDOWS_PRE = {'Ret_7d': 7, 'Ret_14d': 14, 'Ret_21d': 21}

pre_ret_panels = {}   # {'Ret_7d': DataFrame(quarters × tickers), ...}

for label, n_days in WINDOWS_PRE.items():
    panel_ret = pd.DataFrame(index=eps_panel.index, columns=TOP10, dtype=float)
    for ticker in TOP10:
        if ticker not in prices.columns:
            continue
        for q in eps_panel.index:
            ann_date = dates_panel.loc[q, ticker]
            if pd.isna(ann_date):
                continue
            ann_date = pd.Timestamp(ann_date)
            avail = prices.index[prices.index < ann_date]   # strictly before announcement
            if len(avail) < n_days + 1:
                continue
            t_end   = avail[-1]           # day before announcement
            t_start = avail[-n_days]      # n_days back
            panel_ret.loc[q, ticker] = float(
                log_rets.loc[t_start:t_end, ticker].sum()
            )
    pre_ret_panels[label] = panel_ret.astype(float)

# ── Cross-sectional z-score normalization at each quarter
pre_ret_norm = {}
for label, panel in pre_ret_panels.items():
    norm = panel.apply(lambda row: (row - row.mean()) / row.std(ddof=1)
                       if row.notna().sum() > 2 else row, axis=1)
    pre_ret_norm[label] = norm

print("Pre-earnings return panels built.")
print("\nSample — Ret_7d (last 4 quarters, raw):")
display(pre_ret_panels['Ret_7d'].tail(4).style
        .format("{:.4f}", na_rep="—")
        .background_gradient(cmap='RdYlGn', axis=None)
        .set_caption("<b>7-day pre-earnings raw log returns</b>"))

print("\nSample — Ret_7d (last 4 quarters, cross-sec z-scored):")
display(pre_ret_norm['Ret_7d'].tail(4).style
        .format("{:.3f}", na_rep="—")
        .background_gradient(cmap='RdYlGn', axis=None)
        .set_caption("<b>7-day pre-earnings normalized returns</b>"))

Pre-earnings return panels built.

Sample — Ret_7d (last 4 quarters, raw):


,AAPL,MSFT,NVDA,AMZN,GOOGL,META,TSLA,AVGO,LLY,JPM
Quarter (0=latest),,,,,,,,,,
-3,0.0658,0.0959,-0.0044,0.0937,0.0013,0.1247,-0.0589,0.0981,-0.0412,-0.0338
-2,-0.0324,0.0062,-0.0023,0.0288,0.0466,-0.0252,0.0482,0.0396,-0.1744,-0.0324
-1,0.0323,0.0468,-0.0650,0.0037,0.0679,0.0263,0.0070,0.0630,0.0547,-0.0211
0,0.0459,0.0463,0.0674,-0.0942,-0.0007,0.0753,-0.0139,-0.0396,0.0409,-0.0413



Sample — Ret_7d (last 4 quarters, cross-sec z-scored):


,AAPL,MSFT,NVDA,AMZN,GOOGL,META,TSLA,AVGO,LLY,JPM
Quarter (0=latest),,,,,,,,,,
-3,0.463,0.903,-0.562,0.871,-0.479,1.323,-1.359,0.934,-1.101,-0.992
-2,-0.343,0.242,0.113,0.584,0.855,-0.235,0.879,0.747,-2.497,-0.345
-1,0.259,0.607,-2.081,-0.428,1.114,0.114,-0.349,0.995,0.796,-1.026
0,0.671,0.678,1.060,-1.852,-0.167,1.201,-0.406,-0.868,0.582,-0.900


In [34]:
# ── Regression: SUE ~ Pre-earnings return  (pooled panel + Fama-MacBeth)
# For each window, run:
#   1. Pooled OLS (all quarters × stocks stacked)
#   2. Fama-MacBeth: cross-sectional OLS per quarter → mean β, Newey-West t-stat

def pool_data(sue_panel, ret_panel):
    """Stack SUE and return into aligned vectors, drop NaNs."""
    rows = []
    for q in sue_panel.index:
        for ticker in sue_panel.columns:
            s = sue_panel.loc[q, ticker]
            r = ret_panel.loc[q, ticker] if q in ret_panel.index else np.nan
            if pd.notna(s) and pd.notna(r):
                rows.append({'quarter': q, 'ticker': ticker, 'SUE': s, 'Ret': r})
    return pd.DataFrame(rows)

results = {}

for label in WINDOWS_PRE:
    ret_panel = pre_ret_norm[label]   # use normalized returns
    df = pool_data(sue_panel, ret_panel)
    if len(df) < 10:
        continue

    # 1. Pooled OLS
    y = df['SUE'].values
    X = sm.add_constant(df['Ret'].values)
    pooled = sm.OLS(y, X).fit(cov_type='HC3')

    # 2. Fama-MacBeth: per-quarter cross-sectional regression
    betas_fm = []
    for q, grp in df.groupby('quarter'):
        if len(grp) < 4:
            continue
        y_q = grp['SUE'].values
        X_q = sm.add_constant(grp['Ret'].values)
        b = sm.OLS(y_q, X_q).fit().params[1]
        betas_fm.append(b)

    betas_fm = np.array(betas_fm)
    fm_mean  = betas_fm.mean()
    fm_se    = betas_fm.std(ddof=1) / np.sqrt(len(betas_fm))
    fm_t     = fm_mean / fm_se

    results[label] = {
        'N obs':          len(df),
        'N quarters':     df['quarter'].nunique(),
        'Pooled β':       pooled.params[1],
        'Pooled t':       pooled.tvalues[1],
        'Pooled R²':      pooled.rsquared,
        'FM β (mean)':    fm_mean,
        'FM t-stat':      fm_t,
    }

res_df = pd.DataFrame(results).T
res_df.index.name = 'Window'

print("Regression: SUE ~ Normalized Pre-Earnings Return\n")
display(res_df.style
        .format({'N obs': '{:.0f}', 'N quarters': '{:.0f}',
                 'Pooled β': '{:.4f}', 'Pooled t': '{:.3f}', 'Pooled R²': '{:.4f}',
                 'FM β (mean)': '{:.4f}', 'FM t-stat': '{:.3f}'})
        .background_gradient(subset=['Pooled R²'], cmap='YlGn')
        .background_gradient(subset=['FM t-stat'], cmap='YlGn')
        .set_caption("<b>Which pre-earnings window predicts SUE?  (|t| > 1.96 = significant)</b>"))

Regression: SUE ~ Normalized Pre-Earnings Return



,N obs,N quarters,Pooled β,Pooled t,Pooled R²,FM β (mean),FM t-stat
Window,,,,,,,
Ret_7d,160,16,0.3563,1.382,0.0187,0.3563,1.669
Ret_14d,160,16,0.3862,1.665,0.0219,0.3862,1.990
Ret_21d,160,16,0.1389,0.854,0.0028,0.1389,0.718
